In [74]:
import os
import time
import random
import mlflow 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error


if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

# device = torch.device("cpu")  # Force CPU for testing time
random_seed = 42


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("z_prediction")
mlflow.enable_system_metrics_logging()


## Use Hugos functions

In [75]:
def split_csvfiles(datafolder, random_seed):
    csv_files = []
    for f in os.listdir(datafolder):
        if f.endswith(".csv"):
            csv_files.append(f)
    
    random.seed(random_seed)
    random.shuffle(csv_files)

    # Train/Validation/Test with proportions 80/10/10
    train_n = int(len(csv_files) * 0.8)
    val_n = int(len(csv_files) * 0.1)
    test_n = len(csv_files) - train_n - val_n

    # Split
    train_files = csv_files[:train_n]
    val_files = csv_files[train_n: train_n + val_n]
    test_files = csv_files[train_n + test_n:]

    return train_files, val_files, test_files

def load(files, data_dir):
    dataframes = []

    for f in files:
        path = os.path.join(data_dir, f)   
        df = pd.read_csv(path)           
        dataframes.append(df)             

    combined = pd.concat(dataframes, ignore_index=True)  # combine all

    return combined


def input_target_split(dataframe):
    input_cols = []
    target_cols = []
    for c in dataframe.columns:
        if c.endswith("_x") or c.endswith("_y"):
            input_cols.append(c)
        if c.endswith("_z"):
            target_cols.append(c)
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

## Load in the data

In [76]:
datafolder = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
random_seed = 42

train_files, val_files, test_files = split_csvfiles(datafolder, random_seed)

train_data = load(train_files, datafolder)
val_data = load(val_files, datafolder)
test_data = load(test_files, datafolder)

x_train, y_train = input_target_split(train_data)
x_val, y_val = input_target_split(val_data)
x_test, y_test = input_target_split(test_data)


x_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
x_val = torch.tensor(x_val.values, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val.values, dtype=torch.float32).to(device)
x_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Smaller dataset to "force" overfitting
"""
x_train = x_train[:50]
y_train = y_train[:50]
"""

'\nx_train = x_train[:50]\ny_train = y_train[:50]\n'

## Train the model

Hyperparameters and what we do with them: 

* "epochs"

* "optimizer": (SGD, RMSprop, Adam, …)

* loss functition: MSE, MAE 

* model_type

* Hidden layers: One or several hidden layers with varying  number of nodes, Variants of layer types, Activation and Initialization


* "activation": Video said ReLU was good choice so this is not a function that we are super intrested in.



In [77]:
def init_weights(m):
        if isinstance(m, nn.Linear):
            #nn.init.xavier_uniform_(m.weight)  # good for Tanh
            nn.init.kaiming_uniform_(m.weight) # good for ReLU
            nn.init.zeros_(m.bias)

    
class ZPredictor(nn.Module):
    def __init__(self, hidden_layers: list, activation=nn.ReLU(), dropout=0.0):
        super().__init__()
        
        layers = []
        input_size = 26  # 13 joints x 2 (x, y)
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(activation)
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_size = hidden_size
        
        layers.append(nn.Linear(input_size, 13))  # 13 joints z output
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)
    





In [80]:
params = {
    "hidden_layers": [256, 128, 128, 64, 64],
    "activation": nn.ReLU(),
    "dropout": 0,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 100,
    "run_name": "test_speed_gpu",
    #"patience": 5
}

model = ZPredictor(hidden_layers=params["hidden_layers"], activation=params["activation"], dropout=params["dropout"]).to(device)
model.apply(init_weights) 

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

In [81]:
with mlflow.start_run(run_name=params["run_name"]) as run:
    start_time = time.perf_counter()
    mlflow.log_params(params)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(params["epochs"]):

        # Train step
        model.train()
        optimizer.zero_grad()
        predictions = model(x_train)
        train_loss = loss_fn(predictions, y_train)
        train_loss.backward()
        optimizer.step()

        # Evaluation step
        model.eval()
        with torch.no_grad():
            val_predictions = model(x_val)
            val_loss = loss_fn(val_predictions, y_val)
            val_mae = torch.mean(torch.abs(val_predictions - y_val))
            val_r2 = r2_score(y_val.cpu(), val_predictions.cpu())
            val_bias = torch.mean(val_predictions - y_val)

            train_predictions = model(x_train)
            train_mae = torch.mean(torch.abs(train_predictions - y_train))
            train_r2 = r2_score(y_train.cpu(), train_predictions.cpu())
            train_bias = torch.mean(train_predictions - y_train)

        # Log per epoch metrics with step
        mlflow.log_metrics({
            "train_loss": train_loss.item(),
            "train_mae": train_mae.item(),
            "train_r2": train_r2,
            "train_bias": train_bias.item(),
            "val_loss": val_loss.item(),
            "val_mae": val_mae.item(),
            "val_r2": val_r2,
            "val_bias": val_bias.item()
        }, step=epoch)

        print(f"Epoch {epoch+1}/{params['epochs']} "
              f"train_loss: {train_loss.item():.4f} "
              f"train_mae: {train_mae.item():.4f} "
              f"val_loss: {val_loss.item():.4f} "
              f"val_mae: {val_mae.item():.4f} cm")

        # Early stopping
        """
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            mlflow.pytorch.log_model(model, name="best_model")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= params["patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break
        """

    # Test evaluation — only once after training, no step
    model.eval()
    with torch.no_grad():
        test_predictions = model(x_test)
        test_loss = loss_fn(test_predictions, y_test)
        test_mae = torch.mean(torch.abs(test_predictions - y_test))
        test_r2 = r2_score(y_test.cpu(), test_predictions.cpu())
        test_bias = torch.mean(test_predictions - y_test)

    time_elapsed = time.perf_counter() - start_time
    mlflow.log_metrics({
        "test_loss": test_loss.item(),
        "test_mae": test_mae.item(),
        "test_r2": test_r2,
        "test_bias": test_bias.item(),
        "time_seconds": time_elapsed    
    })

    mlflow.pytorch.log_model(model, name=params["run_name"])

2026/04/12 18:42:51 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/12 18:42:51 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/100 train_loss: 1.4442 train_mae: 0.6562 val_loss: 0.5678 val_mae: 0.6202 cm
Epoch 2/100 train_loss: 0.6319 train_mae: 0.3855 val_loss: 0.2198 val_mae: 0.3629 cm
Epoch 3/100 train_loss: 0.2447 train_mae: 0.2519 val_loss: 0.0943 val_mae: 0.2374 cm
Epoch 4/100 train_loss: 0.1064 train_mae: 0.1851 val_loss: 0.0477 val_mae: 0.1763 cm
Epoch 5/100 train_loss: 0.0540 train_mae: 0.1559 val_loss: 0.0327 val_mae: 0.1516 cm
Epoch 6/100 train_loss: 0.0362 train_mae: 0.1438 val_loss: 0.0285 val_mae: 0.1420 cm
Epoch 7/100 train_loss: 0.0302 train_mae: 0.1410 val_loss: 0.0271 val_mae: 0.1387 cm
Epoch 8/100 train_loss: 0.0282 train_mae: 0.1328 val_loss: 0.0250 val_mae: 0.1304 cm
Epoch 9/100 train_loss: 0.0259 train_mae: 0.1237 val_loss: 0.0225 val_mae: 0.1196 cm
Epoch 10/100 train_loss: 0.0235 train_mae: 0.1150 val_loss: 0.0196 val_mae: 0.1098 cm
Epoch 11/100 train_loss: 0.0207 train_mae: 0.1061 val_loss: 0.0168 val_mae: 0.1006 cm
Epoch 12/100 train_loss: 0.0178 train_mae: 0.0975 val_loss: 0.0

2026/04/12 18:42:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Epoch 96/100 train_loss: 0.0049 train_mae: 0.0524 val_loss: 0.0060 val_mae: 0.0556 cm
Epoch 97/100 train_loss: 0.0049 train_mae: 0.0523 val_loss: 0.0060 val_mae: 0.0555 cm
Epoch 98/100 train_loss: 0.0049 train_mae: 0.0522 val_loss: 0.0060 val_mae: 0.0553 cm
Epoch 99/100 train_loss: 0.0049 train_mae: 0.0520 val_loss: 0.0060 val_mae: 0.0552 cm
Epoch 100/100 train_loss: 0.0048 train_mae: 0.0519 val_loss: 0.0060 val_mae: 0.0551 cm


2026/04/12 18:42:56 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/12 18:42:56 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run test_speed_gpu at: http://127.0.0.1:5000/#/experiments/1/runs/f8eaaab0ba0e405cbfdb18e439a508d5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
